# 09 Enterprise NLP Applications & End-to-End Production Pipelines

**Real-World Scenario**: Enterprise Infrastructure Incident Classification & Automated Resolution Pipeline.

This notebook demonstrates building an end-to-end production NLP microservice class, saving pipeline artifacts into a hidden `.model_cache/` directory, computing QPS capacity, and executing a **Complete GenAI Microservice Synthesis LLM Call**.

## Step 1: Environment Setup & Local Cache Configuration

In [ ]:
import os
from dotenv import load_dotenv

# Load API keys from root .env file
load_dotenv()

# Create hidden model cache directory for local pipeline artifact persistence
os.makedirs(".model_cache", exist_ok=True)

print("=== Environment Setup ===")
print("OPENAI_API_KEY present:", bool(os.getenv("OPENAI_API_KEY")))
print("GEMINI_API_KEY present:", bool(os.getenv("GEMINI_API_KEY")))
print("Hidden Cache Directory Path:", os.path.abspath(".model_cache"))

## Step 2: Step-by-Step Production Server QPS Capacity Calculation

We calculate server throughput for a 4 vCPU web server with $20\text{ms}$ inference latency per query.

In [ ]:
latency_per_request_ms = 20.0
vcpus = 4
scaling_efficiency = 0.85

qps_single = 1000.0 / latency_per_request_ms
qps_total = vcpus * qps_single * scaling_efficiency
daily_capacity = qps_total * 86400

print("=== Production Server Capacity Calculation ===")
print(f"Single-Thread QPS:           {qps_single:.1f} req/sec")
print(f"Total 4 vCPU Server QPS:     {qps_total:.1f} req/sec")
print(f"Estimated Daily Capacity:    {daily_capacity:,.0f} requests/day")

## Step 3: End-to-End Enterprise NLP Microservice Implementation & Artifact Persistence

In [ ]:
import joblib
import json
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

class EnterpriseNLPService:
    def __init__(self, model_dir: str = ".model_cache"):
        self.model_dir = model_dir
        self.model_path = os.path.join(self.model_dir, "microservice_pipeline.pkl")
        self.pipeline = self._initialize_model()
        
    def _initialize_model(self):
        os.makedirs(self.model_dir, exist_ok=True)
        X = [
            "Database server 10.0.1.4 error connection timeout",
            "Billing invoice payment failed 402 declined",
            "PostgreSQL replica failover script execution error"
        ]
        y = ["Infrastructure", "Billing", "Infrastructure"]
        
        pipe = Pipeline([
            ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
            ("clf", LogisticRegression())
        ])
        pipe.fit(X, y)
        joblib.dump(pipe, self.model_path)
        return pipe

    def predict_payload(self, raw_json: str) -> str:
        payload = json.loads(raw_json)
        text = payload.get("text", "").strip()
        if not text:
            return json.dumps({"error": "Empty text", "status": 400})
            
        pred = self.pipeline.predict([text])[0]
        conf = float(max(self.pipeline.predict_proba([text])[0]))
        return json.dumps({"text": text, "category": pred, "confidence": round(conf, 4), "status": 200}, indent=2)

service = EnterpriseNLPService()
print(f"=== Production Service Artifact Saved to Hidden Folder: {service.model_path} ===")

## Step 4: Microservice Inference API Execution

In [ ]:
request_json = json.dumps({"text": "PostgreSQL primary database failover script connection timeout"})
response_json = service.predict_payload(request_json)

print("=== Production Pipeline Response Output ===")
print(response_json)

## Step 5: Executing Complete GenAI Production Microservice LLM Call

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

res_dict = json.loads(response_json)

if os.getenv("OPENAI_API_KEY"):
    prompt = ChatPromptTemplate.from_template('''You are an Enterprise Production Service AI.
A production pipeline classified incident: '{text}' as Category: '{cat}' with confidence {conf:.2f}.
Synthesize a complete production incident resolution ticket response.

Incident Resolution Response:''')
    
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)
    
    response = llm.invoke(prompt.format(text=res_dict['text'], cat=res_dict['category'], conf=res_dict['confidence']))
    print("=== Complete GenAI Production Microservice Response ===")
    print(response.content)
else:
    print("[NOTICE] OPENAI_API_KEY not found in .env. Skipping live LLM call.")